## 2.1 理论计算题

**问题描述:**
输入一张大小为 3 × 32 × 32（通道数 × 高 × 宽）的彩色图像。通过一个卷积层，该层包含 16 个卷积核，每个卷积核的大小为 3 × 5 × 5。设定填充（Padding）为 2，步幅（Stride）为 2。

### 问题 1: 计算该卷积层输出的特征图尺寸（通道数 × 高 × 宽）

**解答:**

卷积输出尺寸公式:
$$H_{out} = \lfloor \frac{H_{in} + 2 \times padding - kernel\_size}{stride} \rfloor + 1$$
$$W_{out} = \lfloor \frac{W_{in} + 2 \times padding - kernel\_size}{stride} \rfloor + 1$$

已知参数:
- 输入尺寸: $C_{in} \times H_{in} \times W_{in} = 3 \times 32 \times 32$
- 卷积核数量: 16
- 卷积核尺寸: $C_{in} \times K_h \times K_w = 3 \times 5 \times 5$
- Padding = 2
- Stride = 2

计算过程:
- 输出通道数 $C_{out} = 16$
- 输出高度: $H_{out} = \lfloor \frac{32 + 2 \times 2 - 5}{2} \rfloor + 1 = \lfloor \frac{31}{2} \rfloor + 1 = 15 + 1 = 16$
- 输出宽度: $W_{out} = \lfloor \frac{32 + 2 \times 2 - 5}{2} \rfloor + 1 = \lfloor \frac{31}{2} \rfloor + 1 = 15 + 1 = 16$

**答案: 输出特征图尺寸为 16 × 16 × 16**

### 问题 2: 计算单个输出通道的一个像素值需要对输入进行多少次点乘操作？

**解答:**

对于单个输出通道的一个像素:
1. 卷积核覆盖输入的一个局部区域
2. 区域大小等于卷积核大小: $3 \times 5 \times 5$
3. 卷积执行逐元素乘法然后求和

计算过程:
- 卷积核元素数量 = $3 \times 5 \times 5 = 75$
- 每个元素对应一次乘法操作

**答案: 75 次点乘操作**

## 2.2 编程题

**问题描述:**
不使用深度学习框架的底层 Pooling API（如 `torch.nn.MaxPool2d`），仅使用 Python 和 NumPy（或 PyTorch 基础张量操作），手动实现一个支持步幅（stride）和填充（padding）的二维最大池化（Max Pooling）前向传播函数。

In [108]:
import numpy as np

def max_pool2d_forward(x, kernel_size, stride=1, padding=0):
    """
    手动实现二维最大池化前向传播
    
    参数:
        x: 输入张量，形状为 (C, H, W) 或 (N, C, H, W)
        kernel_size: 池化核大小，可以是整数或元组 (kh, kw)
        stride: 步幅，可以是整数或元组 (sh, sw)，默认为 1
        padding: 填充，可以是整数或元组 (ph, pw)，默认为 0
    
    返回:
        输出张量
    """
    # 处理输入维度
    if len(x.shape) == 3:
        x = x[np.newaxis, :]
        squeeze_output = True
    else:
        squeeze_output = False
    
    N, C, H, W = x.shape
    
    # 处理 kernel_size, stride, padding 参数
    if isinstance(kernel_size, int):
        kh, kw = kernel_size, kernel_size
    else:
        kh, kw = kernel_size
    
    if isinstance(stride, int):
        sh, sw = stride, stride
    else:
        sh, sw = stride
    
    if isinstance(padding, int):
        ph, pw = padding, padding
    else:
        ph, pw = padding
    
    # 填充输入，先转换为 float 以支持 -inf 填充
    x_float = x.astype(np.float64)
    if ph > 0 or pw > 0:
        x_padded = np.pad(x_float, 
                         ((0, 0), (0, 0), (ph, ph), (pw, pw)),
                         mode='constant',
                         constant_values=-np.inf)
    else:
        x_padded = x_float
    
    # 计算输出尺寸
    H_padded, W_padded = x_padded.shape[2], x_padded.shape[3]
    H_out = (H_padded - kh) // sh + 1
    W_out = (W_padded - kw) // sw + 1
    
    # 初始化输出
    output = np.zeros((N, C, H_out, W_out))
    
    # 执行最大池化
    for n in range(N):
        for c in range(C):
            for i in range(H_out):
                for j in range(W_out):
                    h_start = i * sh
                    h_end = h_start + kh
                    w_start = j * sw
                    w_end = w_start + kw
                    
                    window = x_padded[n, c, h_start:h_end, w_start:w_end]
                    output[n, c, i, j] = np.max(window)
    
    # 如果输入是 3D，移除 batch 维度
    if squeeze_output:
        output = output[0]
    
    return output

### 测试函数

In [109]:
# 测试用例 1: 基础测试
print("测试用例 1: 基础测试")
x1 = np.array([[[1, 2, 3, 4],
                [5, 6, 7, 8],
                [9, 10, 11, 12],
                [13, 14, 15, 16]]])

output1 = max_pool2d_forward(x1, kernel_size=2, stride=2, padding=0)
print("输入 shape:", x1.shape)
print("输出 shape:", output1.shape)
print("输出:")
print(output1)
print()

测试用例 1: 基础测试
输入 shape: (1, 4, 4)
输出 shape: (1, 2, 2)
输出:
[[[ 6.  8.]
  [14. 16.]]]



In [110]:
# 测试用例 2: 填充测试
print("测试用例 2: 填充测试")
x2 = np.array([[[1, 2, 3],
                [4, 5, 6],
                [7, 8, 9]]])

output2 = max_pool2d_forward(x2, kernel_size=2, stride=1, padding=1)
print("输入 shape:", x2.shape)
print("输出 shape:", output2.shape)
print("输出:")
print(output2)
print()

测试用例 2: 填充测试
输入 shape: (1, 3, 3)
输出 shape: (1, 4, 4)
输出:
[[[1. 2. 3. 3.]
  [4. 5. 6. 6.]
  [7. 8. 9. 9.]
  [7. 8. 9. 9.]]]



In [111]:
# 测试用例 3: 多通道测试
print("测试用例 3: 多通道测试")
x3 = np.array([[[1, 2, 3, 4],
                 [5, 6, 7, 8]],
                [[9, 10, 11, 12],
                 [13, 14, 15, 16]]])

output3 = max_pool2d_forward(x3, kernel_size=2, stride=2, padding=0)
print("输入 shape:", x3.shape)
print("输出 shape:", output3.shape)
print("输出:")
print(output3)
print()

测试用例 3: 多通道测试
输入 shape: (2, 2, 4)
输出 shape: (2, 1, 2)
输出:
[[[ 6.  8.]]

 [[14. 16.]]]



In [112]:
# 测试用例 4: 批量测试
print("测试用例 4: 批量测试")
x4 = np.array([[[[1, 2],
                   [3, 4]]],
                 [[[5, 6],
                   [7, 8]]]])

output4 = max_pool2d_forward(x4, kernel_size=2, stride=1, padding=0)
print("输入 shape:", x4.shape)
print("输出 shape:", output4.shape)
print("输出:")
print(output4)
print()

测试用例 4: 批量测试
输入 shape: (2, 1, 2, 2)
输出 shape: (2, 1, 1, 1)
输出:
[[[[4.]]]


 [[[8.]]]]



In [113]:
# 与 PyTorch MaxPool2d 比较
print("与 PyTorch MaxPool2d 比较")
try:
    import torch
    import torch.nn as nn
    
    x_test = np.random.randn(2, 3, 8, 8)
    
    our_output = max_pool2d_forward(x_test, kernel_size=3, stride=2, padding=1)
    
    x_torch = torch.from_numpy(x_test).float()
    maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
    torch_output = maxpool(x_torch).numpy()
    
    print("自定义实现输出 shape:", our_output.shape)
    print("PyTorch 输出 shape:", torch_output.shape)
    print("最大差异:", np.max(np.abs(our_output - torch_output)))
    print("结果匹配:", np.allclose(our_output, torch_output))
except ImportError:
    print("PyTorch 未安装，跳过比较")

与 PyTorch MaxPool2d 比较
自定义实现输出 shape: (2, 3, 4, 4)
PyTorch 输出 shape: (2, 3, 4, 4)
最大差异: 1.1271804023493814e-07
结果匹配: True


## 3.1 理论计算题

**问题描述:**
在 VGG 网络中，作者频繁使用多个 3 × 3 卷积核级联来代替较大的卷积核（如 5 × 5 或 7 × 7）。假设输入和输出的特征图通道数均为 C。

### 问题 1: 计算一个 5 × 5 卷积层（不带偏置）的参数量

**解答:**

卷积参数量公式（不带偏置）:
- 单个卷积核参数 = 输入通道数 × 卷积核高度 × 卷积核宽度
- 总参数 = 卷积核数量 × 单个卷积核参数

已知:
- 输入通道数 = C
- 输出通道数 = C（卷积核数量 = C）
- 卷积核尺寸 = 5 × 5

计算过程:
- 单个卷积核参数 = C × 5 × 5 = 25C
- 总参数 = C × 25C = 25C²

**答案: 一个 5 × 5 卷积层的参数量 = 25C²**

### 问题 2: 计算两个串联的 3 × 3 卷积层（不带偏置，两层通道数都为 C）的总参数量

**解答:**

两个串联的 3 × 3 卷积层:
- 第一层: 输入通道数 = C, 输出通道数 = C
- 第二层: 输入通道数 = C, 输出通道数 = C

第一层参数:
- 单个卷积核参数 = C × 3 × 3 = 9C
- 第一层总参数 = C × 9C = 9C²

第二层参数:
- 单个卷积核参数 = C × 3 × 3 = 9C
- 第二层总参数 = C × 9C = 9C²

总参数 = 9C² + 9C² = 18C²

**答案: 两个串联的 3 × 3 卷积层总参数量 = 18C²**

**结论:**
使用两个 3 × 3 层（18C² 参数）代替一个 5 × 5 层（25C² 参数）可以减少约 28% 的参数，同时保持相同的感受野。

## 3.2 编程题

**问题描述:**
NiN 网络的核心创新是引入了"1x1 卷积"组成的 NiN 块来代替传统的全连接层，以减少参数量。请使用 PyTorch（`torch.nn.Sequential`）定义一个标准的 NiN 块（NiN Block）。

要求: NiN 块接收输入通道数 `in_channels` 和输出通道数 `out_channels`，它由一个普通的卷积层（指定窗口大小 `kernel_size`，步幅 `stride`，填充 `padding`）以及两个随后的 1 × 1 卷积层级联组成。每层卷积后都需要紧跟一个 ReLU 激活层。

In [114]:
import torch
import torch.nn as nn

def nin_block(in_channels, out_channels, kernel_size, stride, padding):
    """
    定义标准的 NiN 块
    
    参数:
        in_channels: 输入通道数
        out_channels: 输出通道数
        kernel_size: 卷积核大小
        stride: 步幅
        padding: 填充
    
    返回:
        NiN 块 (torch.nn.Sequential)
    """
    return nn.Sequential(
        nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding),
        nn.ReLU(),
        nn.Conv2d(out_channels, out_channels, kernel_size=1),
        nn.ReLU(),
        nn.Conv2d(out_channels, out_channels, kernel_size=1),
        nn.ReLU()
    )

### 测试 NiN 块

In [115]:
# 测试用例: 创建并测试 NiN 块
print("测试 NiN 块")

# 创建 NiN 块: 输入 3 通道, 输出 16 通道, 卷积核 3x3, 步幅 1, 填充 1
block = nin_block(in_channels=3, out_channels=16, kernel_size=3, stride=1, padding=1)
print("NiN 块结构:")
print(block)
print()

# 创建随机输入张量 (batch_size=2, channels=3, height=32, width=32)
x = torch.randn(2, 3, 32, 32)
print("输入 shape:", x.shape)

# 通过 NiN 块
output = block(x)
print("输出 shape:", output.shape)

# 验证输出通道数
assert output.shape[1] == 16, f"期望输出通道数 16, 得到 {output.shape[1]}"
print("测试通过!")

测试 NiN 块
NiN 块结构:
Sequential(
  (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (1): ReLU()
  (2): Conv2d(16, 16, kernel_size=(1, 1), stride=(1, 1))
  (3): ReLU()
  (4): Conv2d(16, 16, kernel_size=(1, 1), stride=(1, 1))
  (5): ReLU()
)

输入 shape: torch.Size([2, 3, 32, 32])
输出 shape: torch.Size([2, 16, 32, 32])
测试通过!


## 4.1 理论计算题

**问题描述:**
在一个小批量（Mini-batch）训练中，某一个通道内某一特定空间位置的特征值在 4 个样本上的输出分别为：x1 = 2, x2 = 4, x3 = 6, x4 = 8。假设当前批量归一化层学到的缩放参数 γ = 2，平移参数 β = 1，常数 ε = 0。请计算这 4 个样本经由该 Batch Normalization 层转化后的最终输出值 y1, y2, y3, y4。

### 解答

批量归一化公式:

1. 计算批量均值:
$$\mu_B = \frac{1}{m} \sum_{i=1}^{m} x_i$$

2. 计算批量方差:
$$\sigma_B^2 = \frac{1}{m} \sum_{i=1}^{m} (x_i - \mu_B)^2$$

3. 归一化:
$${\hat{x}}_i = \frac{x_i - \mu_B}{\sqrt{\sigma_B^2 + \epsilon}}$$

4. 缩放和平移:
$$y_i = \gamma \cdot {\hat{x}}_i + \beta$$

**已知:**
- x1 = 2, x2 = 4, x3 = 6, x4 = 8
- γ = 2, β = 1, ε = 0
- m = 4

**计算过程:**

1. 批量均值:
$$\mu_B = \frac{2 + 4 + 6 + 8}{4} = \frac{20}{4} = 5$$

2. 批量方差:
$$\sigma_B^2 = \frac{(2-5)^2 + (4-5)^2 + (6-5)^2 + (8-5)^2}{4}$$
$$= \frac{9 + 1 + 1 + 9}{4} = \frac{20}{4} = 5$$

3. 标准差:
$$\sqrt{\sigma_B^2 + \epsilon} = \sqrt{5 + 0} = \sqrt{5} \approx 2.236$$

4. 归一化:
$${\hat{x}}_1 = \frac{2 - 5}{\sqrt{5}} = \frac{-3}{\sqrt{5}} \approx -1.3416$$
$${\hat{x}}_2 = \frac{4 - 5}{\sqrt{5}} = \frac{-1}{\sqrt{5}} \approx -0.4472$$
$${\hat{x}}_3 = \frac{6 - 5}{\sqrt{5}} = \frac{1}{\sqrt{5}} \approx 0.4472$$
$${\hat{x}}_4 = \frac{8 - 5}{\sqrt{5}} = \frac{3}{\sqrt{5}} \approx 1.3416$$

5. 缩放和平移 (γ=2, β=1):
$$y_1 = 2 \cdot \left(\frac{-3}{\sqrt{5}}\right) + 1 = 1 - \frac{6}{\sqrt{5}} \approx -1.683$$
$$y_2 = 2 \cdot \left(\frac{-1}{\sqrt{5}}\right) + 1 = 1 - \frac{2}{\sqrt{5}} \approx 0.106$$
$$y_3 = 2 \cdot \left(\frac{1}{\sqrt{5}}\right) + 1 = 1 + \frac{2}{\sqrt{5}} \approx 1.894$$
$$y_4 = 2 \cdot \left(\frac{3}{\sqrt{5}}\right) + 1 = 1 + \frac{6}{\sqrt{5}} \approx 3.683$$

**答案:**
- y1 = 1 - 6/√5 ≈ -1.683
- y2 = 1 - 2/√5 ≈ 0.106
- y3 = 1 + 2/√5 ≈ 1.894
- y4 = 1 + 6/√5 ≈ 3.683

## 4.2 编程题

**问题描述:**
残差网络（ResNet）通过引入跨层连接（残差连接）解决了深层网络的梯度消失问题。请用 PyTorch 自定义一个残差块类 `Residual`。

In [116]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class Residual(nn.Module):
    """
    自定义残差块类
    
    参数:
        in_channels: 输入通道数
        out_channels: 输出通道数
        stride: 步幅，默认为 1
        use_1x1conv: 是否使用 1x1 卷积进行通道匹配，默认为 False
    """
    def __init__(self, in_channels, out_channels, stride=1, use_1x1conv=False):
        super(Residual, self).__init__()
        
        # 主路径: 两个 3x3 卷积
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, 
                               stride=stride, padding=1)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3,
                               stride=1, padding=1)
        self.bn2 = nn.BatchNorm2d(out_channels)
        
        # 捷径路径: 当通道数或步幅改变时使用 1x1 卷积
        self.conv3 = None
        if use_1x1conv:
            self.conv3 = nn.Conv2d(in_channels, out_channels, kernel_size=1,
                                   stride=stride)
    
    def forward(self, X):
        """
        前向传播
        
        参数:
            X: 输入张量
        
        返回:
            残差块输出
        """
        # 主路径
        Y = F.relu(self.bn1(self.conv1(X)))
        Y = self.bn2(self.conv2(Y))
        
        # 捷径路径
        if self.conv3:
            X = self.conv3(X)
        
        # 残差连接: 主路径 + 捷径路径
        Y += X
        return F.relu(Y)

### 测试残差块

In [117]:
# 测试用例 1: 通道数不变的残差块
print("测试用例 1: 通道数不变的残差块")
residual1 = Residual(in_channels=3, out_channels=3)
print("残差块结构:")
print(residual1)
print()

# 创建输入张量 (batch_size=2, channels=3, height=32, width=32)
x1 = torch.randn(2, 3, 32, 32)
print("输入 shape:", x1.shape)

# 通过残差块
output1 = residual1(x1)
print("输出 shape:", output1.shape)

# 验证输出形状与输入相同
assert output1.shape == x1.shape, f"期望输出形状与输入相同, 得到 {output1.shape}"
print("测试通过!")
print()

测试用例 1: 通道数不变的残差块
残差块结构:
Residual(
  (conv1): Conv2d(3, 3, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn1): BatchNorm2d(3, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (conv2): Conv2d(3, 3, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn2): BatchNorm2d(3, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
)

输入 shape: torch.Size([2, 3, 32, 32])
输出 shape: torch.Size([2, 3, 32, 32])
测试通过!



In [118]:
# 测试用例 2: 通道数改变的残差块
print("测试用例 2: 通道数改变的残差块")
residual2 = Residual(in_channels=3, out_channels=6, stride=2, use_1x1conv=True)
print("残差块结构:")
print(residual2)
print()

# 创建输入张量 (batch_size=2, channels=3, height=32, width=32)
x2 = torch.randn(2, 3, 32, 32)
print("输入 shape:", x2.shape)

# 通过残差块
output2 = residual2(x2)
print("输出 shape:", output2.shape)

# 验证输出通道数变为 6, 尺寸减半
assert output2.shape[1] == 6, f"期望输出通道数 6, 得到 {output2.shape[1]}"
assert output2.shape[2] == 16, f"期望输出高度 16, 得到 {output2.shape[2]}"
assert output2.shape[3] == 16, f"期望输出宽度 16, 得到 {output2.shape[3]}"
print("测试通过!")

测试用例 2: 通道数改变的残差块
残差块结构:
Residual(
  (conv1): Conv2d(3, 6, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
  (bn1): BatchNorm2d(6, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (conv2): Conv2d(6, 6, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn2): BatchNorm2d(6, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (conv3): Conv2d(3, 6, kernel_size=(1, 1), stride=(2, 2))
)

输入 shape: torch.Size([2, 3, 32, 32])
输出 shape: torch.Size([2, 6, 16, 16])
测试通过!


## 5.1 理论计算题

**问题描述:**
在微调（Fine-tuning）任务中，我们通常会在一个大型源数据集（如 ImageNet）上预训练好的网络模型基础上，去适应一个新的目标数据集。请回答以下关于微调理论的问题：

### 问题 1: 为什么我们通常对除了最终输出层之外的"底层特征提取层"设置较小的学习率（甚至将其参数固定/冻结），而对新初始化的"顶层输出层"设置较大的学习率？

**解答:**

采用这种策略的原因:

1. **特征可重用性:**
   - 底层网络学习通用的视觉特征（边缘、纹理、简单形状）
   - 这些特征在不同视觉任务之间具有可迁移性
   - 不需要进行显著调整

2. **避免灾难性遗忘:**
   - 底层网络使用大学习率可能会破坏已学习的特征
   - 小学习率保护有用特征不被过度修改

3. **顶层特异性:**
   - 顶层是任务特定的（如分类）
   - 源数据集和目标数据集的输出类别不同
   - 需要重新初始化并使用大学习率快速适应

4. **训练稳定性:**
   - 冻结或低学习率的底层提供稳定的特征表示
   - 只有顶层需要调整，使训练更稳定

**总结:** 保护预训练的底层特征，同时允许顶层快速适应新任务。

### 问题 2: 如果目标数据集非常小，且与源数据集非常相似，我们应该采取什么样的微调策略以防止过拟合？

**解答:**

针对小型相似数据集的策略:

1. **冻结大部分预训练层:**
   - 小数据集缺乏足够的数据来更新大量参数
   - 冻结除顶层外的所有层，只训练新初始化的输出层
   - 称为"特征提取"模式

2. **使用非常小的学习率:**
   - 如果需要微调底层，使用极小的学习率（如 1e-5）
   - 只执行少量 epoch 的微调

3. **增加正则化:**
   - 在输出或分类层添加 Dropout
   - 使用权重衰减

4. **数据增强:**
   - 对小数据集应用强增强
   - 包括随机裁剪、翻转、颜色变换

5. **提前停止:**
   - 监控验证性能
   - 当验证性能不再提升时停止训练

6. **模型集成:**
   - 训练多个模型并集成它们
   - 使用不同的增强策略

7. **策略选择:**
   - 优先只训练输出层
   - 仅在验证性能停滞时才微调中间层

**总结:** 最大化利用预训练知识，最小化参数更新，并通过正则化和增强提高泛化能力。

## 5.2 编程题

**问题描述:**
图像增广能有效增强模型的泛化能力。请利用 `torchvision.transforms` 模块创建一个组合图像增广管道（Pipeline）。

要求:
1. 随机对图像进行裁剪，使其面积比例在 0.08 到 1.0 之间，并将裁剪后的图像缩放到 224 × 224 像素。
2. 拥有 50% 的概率对图像进行水平翻转。
3. 随机改变图像的亮度（Brightness）、对比度（Contrast）和饱和度（Saturation），变化范围设为 0.5。
4. 最终将图像转换为 PyTorch 张量（Tensor）。

In [119]:
import torch
import torchvision.transforms as transforms

# 创建组合图像增广管道
train_transform = transforms.Compose([
    # 1. 随机裁剪: 面积比例 0.08-1.0, 缩放到 224×224
    transforms.RandomResizedCrop(
        size=(224, 224),
        scale=(0.08, 1.0)
    ),
    # 2. 50% 概率水平翻转
    transforms.RandomHorizontalFlip(p=0.5),
    # 3. 随机亮度、对比度、饱和度变化，范围 0.5
    transforms.ColorJitter(
        brightness=0.5,
        contrast=0.5,
        saturation=0.5
    ),
    # 4. 转换为 PyTorch 张量
    transforms.ToTensor()
])

# 可选: 创建验证/测试转换（无增广）
val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor()
])

### 测试图像增广管道

In [120]:
# 测试用例: 测试图像增广管道
print("测试图像增广管道")

# 创建模拟 PIL 图像 (3 通道, 512x512)
try:
    from PIL import Image
    
    # 创建随机图像
    import numpy as np
    img_np = np.random.randint(0, 256, (512, 512, 3), dtype=np.uint8)
    img = Image.fromarray(img_np)
    
    print("原始图像 shape:", img.size)
    
    # 多次应用训练转换以展示随机性
    print("\n应用训练转换（多次应用以展示随机性）:")
    for i in range(3):
        transformed_img = train_transform(img)
        print(f"  转换 {i+1}: shape={transformed_img.shape}, dtype={transformed_img.dtype}")
    
    # 应用验证转换
    print("\n应用验证转换:")
    val_img = val_transform(img)
    print(f"  shape={val_img.shape}, dtype={val_img.dtype}")
    
    print("\n测试通过!")
    
except ImportError:
    print("PIL 未安装，跳过图像测试")
    print("转换管道创建成功")

测试图像增广管道
原始图像 shape: (512, 512)

应用训练转换（多次应用以展示随机性）:
  转换 1: shape=torch.Size([3, 224, 224]), dtype=torch.float32
  转换 2: shape=torch.Size([3, 224, 224]), dtype=torch.float32
  转换 3: shape=torch.Size([3, 224, 224]), dtype=torch.float32

应用验证转换:
  shape=torch.Size([3, 224, 224]), dtype=torch.float32

测试通过!


## 6.1 理论计算题

**问题描述:**
在目标检测中，交并比（IoU）用于衡量预测边界框与真实边界框的重合程度。已知图像中两个边界框（以 `[左上角 x, 左上角 y, 右下角 x, 右下角 y]` 格式表示）：

1. 真实框（Ground Truth）A = [10, 10, 50, 50]
2. 预测框（Prediction Box）B = [30, 30, 70, 70]

请计算边界框 A 和边界框 B 之间的 IoU 准确值。

### 解答

**IoU 公式:**

$$IoU = \frac{\text{交集面积}}{\text{并集面积}} = \frac{\text{Intersection}}{\text{Union}}$$

**计算过程:**

1. **交集:**
   - 交集左上角 x: max(A_x1, B_x1) = max(10, 30) = 30
   - 交集左上角 y: max(A_y1, B_y1) = max(10, 30) = 30
   - 交集右下角 x: min(A_x2, B_x2) = min(50, 70) = 50
   - 交集右下角 y: min(A_y2, B_y2) = min(50, 70) = 50
   - 交集宽度 = 50 - 30 = 20
   - 交集高度 = 50 - 30 = 20
   - 交集面积 = 20 × 20 = 400

2. **各自面积:**
   - 框 A 宽度 = 50 - 10 = 40
   - 框 A 高度 = 50 - 10 = 40
   - 框 A 面积 = 40 × 40 = 1600
   - 框 B 宽度 = 70 - 30 = 40
   - 框 B 高度 = 70 - 30 = 40
   - 框 B 面积 = 40 × 40 = 1600

3. **并集面积:**

$$Union = Area_A + Area_B - Intersection$$
$$Union = 1600 + 1600 - 400 = 2800$$

4. **IoU:**

$$IoU = \frac{400}{2800} = \frac{1}{7} \approx 0.1429$$

**答案:** IoU = 1/7 ≈ 0.1429

## 6.2 编程题

**问题描述:**
在计算机视觉训练技巧中，标签平滑（Label Smoothing）通过防止模型过于自信地预测某些类别来提高泛化性。标准交叉熵使用独热编码（One-hot），若设置平滑因子 ε = 0.1，则对于 K 分类问题，真实标签对应的目标概率从 1 变为 1 − ε，其余错误类别的概率从 0 变为 ε/(K−1)。

请实现一个计算标签平滑后交叉熵损失的函数。

In [121]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class LabelSmoothingCrossEntropy(nn.Module):
    """
    标签平滑交叉熵损失（与 PyTorch 内置实现一致）
    
    参数:
        epsilon: 平滑因子，默认为 0.1
    """
    def __init__(self, epsilon=0.1):
        super(LabelSmoothingCrossEntropy, self).__init__()
        self.epsilon = epsilon
    
    def forward(self, predictions, targets):
        """
        计算标签平滑后的交叉熵损失
        
        参数:
            predictions: 模型输出的 logits，形状为 (N, K)
            targets: 真实标签，形状为 (N,)
        
        返回:
            损失值
        """
        K = predictions.size(-1)
        log_probs = F.log_softmax(predictions, dim=-1)
        
        # 标准交叉熵（仅对真实标签）
        nll_loss = -log_probs.gather(dim=-1, index=targets.unsqueeze(1)).squeeze(1)
        
        # 均匀分布的交叉熵（对所有类别）
        smooth_loss = -log_probs.mean(dim=-1)
        
        # 混合损失：(1-epsilon)*nll_loss + epsilon*smooth_loss
        loss = (1 - self.epsilon) * nll_loss + self.epsilon * smooth_loss
        
        return loss.mean()

def label_smoothing_cross_entropy(predictions, targets, epsilon=0.1):
    """
    函数形式的标签平滑交叉熵损失
    """
    criterion = LabelSmoothingCrossEntropy(epsilon=epsilon)
    return criterion(predictions, targets)

### 测试标签平滑交叉熵

In [122]:
# 测试用例: 测试标签平滑交叉熵损失
print("测试标签平滑交叉熵")

# 创建测试数据
torch.manual_seed(42)
predictions = torch.randn(4, 5)  # batch_size=4, num_classes=5
targets = torch.tensor([0, 2, 1, 4])  # 真实标签

print("输入:")
print(f"  predictions shape: {predictions.shape}")
print(f"  targets: {targets}")

# 使用自定义实现
criterion = LabelSmoothingCrossEntropy(epsilon=0.1)
loss1 = criterion(predictions, targets)

# 使用函数形式
loss2 = label_smoothing_cross_entropy(predictions, targets, epsilon=0.1)

# 使用 PyTorch 内置标签平滑
torch_criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
loss3 = torch_criterion(predictions, targets)

print("\n测试结果:")
print(f"  使用自定义 LabelSmoothingCrossEntropy (epsilon=0.1): {loss1.item():.4f}")
print(f"  使用函数形式 label_smoothing_cross_entropy (epsilon=0.1): {loss2.item():.4f}")
print(f"  使用 PyTorch CrossEntropyLoss (label_smoothing=0.1): {loss3.item():.4f}")

# 测试不同 epsilon 值
print("\n不同 epsilon 值测试:")
for eps in [0.0, 0.1, 0.2]:
    crit = LabelSmoothingCrossEntropy(epsilon=eps)
    loss = crit(predictions, targets)
    print(f"  epsilon={eps}: {loss.item():.4f}")

# 验证与 PyTorch 实现一致性
print(f"\n损失差异: {abs(loss1.item() - loss3.item()):.6f}")
assert torch.allclose(loss1, loss3, rtol=1e-5, atol=1e-6), f"自定义实现与 PyTorch 不一致, 差异: {abs(loss1.item() - loss3.item()):.6f}"
print("测试通过!")

测试标签平滑交叉熵
输入:
  predictions shape: torch.Size([4, 5])
  targets: tensor([0, 2, 1, 4])

测试结果:
  使用自定义 LabelSmoothingCrossEntropy (epsilon=0.1): 1.3942
  使用函数形式 label_smoothing_cross_entropy (epsilon=0.1): 1.3942
  使用 PyTorch CrossEntropyLoss (label_smoothing=0.1): 1.3942

不同 epsilon 值测试:
  epsilon=0.0: 1.3351
  epsilon=0.1: 1.3942
  epsilon=0.2: 1.4532

损失差异: 0.000000
测试通过!
